# 03 · Non-IID heterogeneity sweep  (Phase **P3** — experiment E2)

G4 proved the FL loop is correct on IID data. Now we introduce **heterogeneity** and study how the
regimes behave — this is where the real science (and the motivation for the proposed FedBN method in
P4) begins.

**What this sweeps:** Dirichlet **α ∈ {100, 1.0, 0.5, 0.1}** (IID → severe label skew), comparing:
- **Centralized** — pooled upper bound (α-independent).
- **Local-only** — each client trains alone; reported on the global test (generalization) and its
  own test (in-domain).
- **FedAvg** — standard federation.
- **FedProx** — FedAvg + proximal term (μ), the non-IID-robust baseline.

**Expected story:** FedAvg degrades as α ↓ (client drift); FedProx should be ≥ FedAvg at low α;
local-only's *global*-test accuracy collapses under skew while its *own*-test stays high.

**Resumable:** every (regime, α, seed) result is appended to a JSONL on Drive as it finishes, and
re-running **skips** completed combos — so a Colab disconnect never loses progress. Just re-run.

> Compute: with the defaults below (~8 FL runs + 4 local-only + 1 centralized, 1 seed) expect a few
> hours on a T4. Run it in sittings; it resumes. Scale `SEEDS`/`ALPHAS` up for the final paper.


## 1 · Environment + Drive

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sgogoigh/Satellite-Image-Classification-Fed-Learning.git"
REPO_DIR = "/content/Satellite-Image-Classification-Fed-Learning"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import torch, fedsat
from fedsat.utils import get_device
print("fedsat", fedsat.__version__, "| torch", torch.__version__, "| device:", get_device())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception as e:
    print("Drive not mounted (", e, ") -> falling back to local ./artifacts")
    DRIVE = os.path.join(REPO_DIR, "artifacts")
os.makedirs(DRIVE, exist_ok=True)
print("Artifacts dir:", DRIVE)


## 2 · Base config + sweep knobs

In [ ]:
from fedsat.config import ExperimentConfig
from fedsat.utils import get_device

BASE = ExperimentConfig(
    experiment_name="p3_noniid_sweep",
    dataset="eurosat_rgb", hf_repo="blanchon/EuroSAT_RGB",
    num_clients=10, input_size=64,
    backbone="resnet18", pretrained=True, norm="bn",
    optimizer="sgd", lr=0.01, momentum=0.9, weight_decay=5e-4, batch_size=64,
    num_rounds=20, local_epochs=1, fraction_fit=1.0, num_workers=2,
    max_epochs=40, early_stop_patience=7,
    data_cache_dir=f"{DRIVE}/hf_cache",
    partition_dir=f"{DRIVE}/partitions",
    results_dir=f"{DRIVE}/results",
    device=get_device(),
)

# ---- sweep knobs (scale up for the paper) ----
ALPHAS = [100.0, 1.0, 0.5, 0.1]     # IID -> severe non-IID
SEEDS = [42]                         # add 43, 44 for confidence intervals (multiplies runtime)
NUM_ROUNDS = 20                      # FL rounds per run
LOCAL_EPOCHS = 1                     # client local epochs / round
LOCAL_ONLY_EPOCHS = 12               # per-client budget for the local-only baseline
MU = 0.1                             # FedProx proximal weight
FED_REGIMES = [("fedavg", 0.0), ("fedprox", MU)]

RESULTS_DIR = f"{DRIVE}/results/p3_noniid_sweep"
print(f"sweep: {len(ALPHAS)} alphas x {len(SEEDS)} seeds | regimes: centralized, local_only, fedavg, fedprox")


## 3 · Load EuroSAT (once)

In [ ]:
from fedsat.data import load_eurosat, integrity_gate

hf_ds, class_names, labels = load_eurosat(BASE.hf_repo, cache_dir=BASE.data_cache_dir)
stats = integrity_gate(class_names, labels, expected_classes=BASE.num_classes,
                       expected_total=BASE.expected_total)
print("data ok:", stats["total"], "images |", stats["num_classes"], "classes | hash", stats["data_hash"])


## 4 · Resumable results store

In [ ]:
import os, json
os.makedirs(RESULTS_DIR, exist_ok=True)
STORE = os.path.join(RESULTS_DIR, "results.jsonl")

def load_rows():
    if not os.path.exists(STORE):
        return []
    return [json.loads(l) for l in open(STORE) if l.strip()]

def is_done(regime, alpha, seed):
    return any(r["regime"] == regime and r.get("alpha") == alpha and r.get("seed") == seed
               for r in load_rows())

def append_row(row):
    with open(STORE, "a") as f:
        f.write(json.dumps(row) + "\n")

print("results store:", STORE, "| existing rows:", len(load_rows()))


## 5 · Centralized upper bound (α-independent, run once)

Centralized trains on the pooled data, so it's essentially independent of how the pool is
partitioned — we run it once (on the IID partition) and use it as the reference line.


In [ ]:
from fedsat.regimes import run_centralized
from fedsat.data import build_partition, save_partition, load_partition
from fedsat.utils import set_seed

if not is_done("centralized", None, None):
    set_seed(42)
    cfg_c = BASE.replace(alpha=100.0, seed=42)
    try:
        part_c = load_partition(cfg_c)
    except FileNotFoundError:
        part_c = build_partition(cfg_c, labels, class_names, data_hash=stats["data_hash"])
        save_partition(cfg_c, part_c)
    print("running centralized (upper bound) ...", flush=True)
    _, csum = run_centralized(cfg_c, hf_ds, part_c, class_names)
    append_row({"regime": "centralized", "alpha": None, "seed": None,
                "global_test_accuracy": csum["global_test_accuracy"],
                "global_test_macro_f1": csum["global_test_macro_f1"],
                "grad_steps": csum["grad_steps"]})
    print("centralized global-test acc:", round(csum["global_test_accuracy"], 4))
else:
    print("centralized already done -- skipping")


## 6 · The sweep — local-only / FedAvg / FedProx across α

This is the long cell. It **resumes**: completed (regime, α, seed) combos are skipped, so you can
stop and re-run freely. Progress prints as each run finishes.


In [ ]:
from fedsat.regimes import run_local_only
from fedsat.fl import run_fedavg
from fedsat.data import build_partition, save_partition, load_partition
from fedsat.utils import set_seed
import time

for alpha in ALPHAS:
    for seed in SEEDS:
        cfg = BASE.replace(alpha=alpha, seed=seed)
        try:
            part = load_partition(cfg)
        except FileNotFoundError:
            part = build_partition(cfg, labels, class_names, data_hash=stats["data_hash"])
            save_partition(cfg, part)

        # --- local-only ---
        if not is_done("local_only", alpha, seed):
            set_seed(seed)
            print(f"[alpha={alpha} seed={seed}] local_only ...", flush=True)
            t = time.time()
            lsum = run_local_only(cfg, hf_ds, part, class_names, local_max_epochs=LOCAL_ONLY_EPOCHS)
            append_row({"regime": "local_only", "alpha": alpha, "seed": seed,
                        "global_test_accuracy": lsum["mean_global_test_accuracy"],
                        "worst_global_test_accuracy": lsum["worst_global_test_accuracy"],
                        "own_test_accuracy": lsum["mean_own_test_accuracy"]})
            print(f"    mean global-test={lsum['mean_global_test_accuracy']:.4f}  "
                  f"own-test={lsum['mean_own_test_accuracy']:.4f}  ({time.time()-t:.0f}s)", flush=True)

        # --- FedAvg + FedProx ---
        for name, mu in FED_REGIMES:
            if is_done(name, alpha, seed):
                continue
            set_seed(seed)
            print(f"[alpha={alpha} seed={seed}] {name} (mu={mu}) ...", flush=True)
            t = time.time()
            _, hist, s = run_fedavg(cfg, hf_ds, part, class_names, num_rounds=NUM_ROUNDS,
                                    local_epochs=LOCAL_EPOCHS, fraction_fit=cfg.fraction_fit,
                                    mu=mu, verbose=False)
            append_row({"regime": name, "alpha": alpha, "seed": seed,
                        "global_test_accuracy": s["test_accuracy_at_best"],
                        "best_round": s["best_round"], "epoch_equivalents": s["epoch_equivalents"]})
            print(f"    global-test acc={s['test_accuracy_at_best']:.4f} "
                  f"(best round {s['best_round']})  ({time.time()-t:.0f}s)", flush=True)

print("\nSWEEP COMPLETE. total rows:", len(load_rows()))


## 7 · Aggregate + the heterogeneity curve

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = pd.DataFrame(load_rows())
cent = df[df.regime == "centralized"]["global_test_accuracy"].mean()

fed = df[df.regime.isin(["local_only", "fedavg", "fedprox"])].copy()
agg = (fed.groupby(["regime", "alpha"])["global_test_accuracy"]
          .agg(["mean", "std", "count"]).reset_index())
print("centralized (upper bound):", round(cent, 4))
print(agg.to_string(index=False))

alphas_sorted = sorted(set(fed["alpha"]))
xs = range(len(alphas_sorted))
plt.figure(figsize=(9, 5))
for regime, color in [("fedprox", "#229954"), ("fedavg", "#d35400"), ("local_only", "#8e44ad")]:
    sub = agg[agg.regime == regime].set_index("alpha").reindex(alphas_sorted)
    plt.errorbar(xs, sub["mean"], yerr=sub["std"].fillna(0.0), marker="o", capsize=4,
                 label=regime, color=color)
plt.axhline(cent, ls="--", color="#444", label=f"centralized ({cent:.3f})")
plt.xticks(list(xs), [str(a) for a in alphas_sorted])
plt.xlabel("Dirichlet alpha   (left = severe non-IID, right = IID)")
plt.ylabel("global-test accuracy"); plt.ylim(0, 1)
plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.title("EuroSAT: federated regimes vs data heterogeneity")
plt.tight_layout(); plt.show()


## 8 · Local-only: in-domain vs global generalization + save tables

In [ ]:
lo = df[df.regime == "local_only"].sort_values("alpha")
if not lo.empty:
    print("Local-only: a lone client memorizes its OWN classes but fails to generalize globally.")
    print(lo[["alpha", "seed", "own_test_accuracy", "global_test_accuracy",
              "worst_global_test_accuracy"]].to_string(index=False))

df.to_csv(os.path.join(RESULTS_DIR, "sweep_summary.csv"), index=False)
agg.to_csv(os.path.join(RESULTS_DIR, "sweep_aggregate.csv"), index=False)
print("\nsaved ->", RESULTS_DIR)

# quick sanity read-out of the expected pattern
def acc(regime, a):
    r = agg[(agg.regime == regime) & (agg.alpha == a)]
    return float(r["mean"].iloc[0]) if len(r) else float("nan")
a_lo, a_hi = min(alphas_sorted), max(alphas_sorted)
print(f"\nFedAvg: alpha={a_hi} -> {acc('fedavg', a_hi):.3f} | alpha={a_lo} -> {acc('fedavg', a_lo):.3f} "
      f"(expect a drop as alpha decreases)")
print(f"At alpha={a_lo}: FedProx {acc('fedprox', a_lo):.3f} vs FedAvg {acc('fedavg', a_lo):.3f} "
      f"(FedProx expected >= FedAvg under skew)")


## Done — P3 complete

You now have the **heterogeneity curve** (accuracy vs α) for centralized / local-only / FedAvg /
FedProx, plus the in-domain-vs-global contrast for local-only — the empirical motivation for a
method that handles cross-client shift.

**Next — Phase P4 (`04_proposed_pftl.ipynb`):** the proposed method. Add **FedBN** (keep BatchNorm
local — targets feature shift) and optional personalization, run the BN-policy and sensor-shift
ablations, and test — with confidence intervals over ≥3 seeds — whether it beats FedAvg/FedProx
under heterogeneity. Only claim a win where the CIs separate.
